# Team 04 End-to-End LangGraph Node Notebook

This notebook is the prompt-driven end-to-end harness for the active Team 04 LangGraph runtime.

It is meant to test one agent-node loop from user prompt through tool calls to a final state, while keeping site context notebook-local and inspectable.

## Scope

Use this notebook when you want to test a LangGraph agent-node style loop against real prompts:
1. start from a user prompt and notebook-provided site state
2. let the active planner and supervisor choose the next action
3. run the local tool surface until the loop reaches a final state

This notebook now includes:
- prompt scenarios for one-building and two-building runs
- notebook-local site objects such as streets or alignment guides
- visualization of the current candidate, placed buildings, and graph payload

Keep `tool_dev_mode.ipynb` for deterministic geometry debugging. Use this notebook for prompt-to-final-state behavior.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (
    workspace_root,
    workspace_root.parent,
    workspace_root / "team_04",
    workspace_root.parent / "team_04",
)
TEAM_ROOT = next((path for path in candidate_roots if (path / "agent").exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError(
        "Run this notebook from the workspace root, the team_04 folder, or the team_04/test_notebooks folder."
    )

team_root_str = str(TEAM_ROOT)
if team_root_str not in sys.path:
    sys.path.insert(0, team_root_str)

E2E_OUTPUT_PATH = TEAM_ROOT / "test_notebooks" / "end_to_end_api_agent_output.json"
E2E_OUTPUT_PATH

WindowsPath('C:/Users/baoqt/OneDrive/Documents/GitHub/AIA26_Studio/team_04/test_notebooks/end_to_end_api_agent_output.json')

In [ ]:
from langchain_openai import ChatOpenAI

import plotly.graph_objects as go

from agent.config import load_settings
from agent.decision_engine import OpenAIDecisionEngine, RuleBasedPlanner
from agent.graph import run_agent
from agent.mcp_client import CompositeToolClient, build_default_local_tool_client
from agent.notebook_demo_tools import build_notebook_demo_tool_client
from agent.tool_catalog import ToolCatalog

settings_error = None
try:
    settings = load_settings()
except Exception as exc:
    settings = None
    settings_error = str(exc)

{
    "llm_provider": settings.llm_provider if settings is not None else None,
    "llm_model": settings.llm_model if settings is not None else None,
    "settings_error": settings_error,
}

{'llm_provider': None,
 'llm_model': None,
 'settings_error': 'Missing required environment variable: LLM_PROVIDER'}

In [ ]:
def _to_xyz(point):
    if len(point) >= 3:
        return (float(point[0]), float(point[1]), float(point[2]))
    return (float(point[0]), float(point[1]), 0.0)


def _dedupe_boundary(boundary):
    cleaned = []
    for point in boundary or []:
        xyz = _to_xyz(point)
        if not cleaned or any(abs(cleaned[-1][index] - xyz[index]) > 1e-6 for index in range(3)):
            cleaned.append(xyz)
    if len(cleaned) > 1 and all(abs(cleaned[0][index] - cleaned[-1][index]) <= 1e-6 for index in range(3)):
        cleaned = cleaned[:-1]
    return cleaned


def _normalize_points(points):
    normalized = []
    for point in points or []:
        if isinstance(point, (list, tuple)) and point and isinstance(point[0], (list, tuple)):
            normalized.extend(_normalize_points(point))
            continue
        normalized.append(_to_xyz(point))
    return normalized


def _apply_zoom_to_fit(figure):
    figure.update_yaxes(scaleanchor="x", scaleratio=1, visible=False)
    figure.update_xaxes(visible=False)
    figure.update_layout(
        margin=dict(l=0, r=0, t=48, b=0),
        plot_bgcolor="#f8fafc",
        paper_bgcolor="#f8fafc",
        font=dict(color="#111827", size=13),
    )
    return figure


def _add_boundary_trace(figure, boundary, *, line_color, line_width, fillcolor=None):
    points = _dedupe_boundary(boundary)
    if len(points) < 2:
        return
    xs = [point[0] for point in points] + [points[0][0]]
    ys = [point[1] for point in points] + [points[0][1]]
    figure.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            line=dict(color=line_color, width=line_width),
            fill="toself" if fillcolor else None,
            fillcolor=fillcolor,
            hoverinfo="skip",
            showlegend=False,
        )
    )


def _add_site_object_traces(figure, site_objects):
    palette = {
        "street": "#2563eb",
        "street_centerline": "#2563eb",
        "alignment_guide": "#7c3aed",
        "setback_edge": "#0f766e",
    }
    for item in site_objects or []:
        if not isinstance(item, dict):
            continue
        points = _normalize_points(item.get("points", []))
        if len(points) < 2:
            continue
        color = palette.get(str(item.get("type", "")).strip(), "#64748b")
        label = str(item.get("name", item.get("type", "site_object"))).strip() or "site_object"
        xs = [point[0] for point in points]
        ys = [point[1] for point in points]
        figure.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode="lines+text",
                line=dict(color=color, width=3, dash="dash"),
                text=[label] + [""] * (len(xs) - 1),
                textposition="top center",
                textfont=dict(color=color, size=12),
                hoverinfo="skip",
                showlegend=False,
            )
        )


def _add_centerline_graph_trace(figure, building_graph):
    centerline_graph = (building_graph or {}).get("centerline_graph", {})
    nodes = centerline_graph.get("nodes", [])
    edges = centerline_graph.get("edges", [])
    if not nodes or not edges:
        return

    for edge in edges:
        start = nodes[edge["from_node_index"]]["point"]
        end = nodes[edge["to_node_index"]]["point"]
        figure.add_trace(
            go.Scatter(
                x=[start[0], end[0]],
                y=[start[1], end[1]],
                mode="lines",
                line=dict(color="#dc2626", width=4),
                hoverinfo="skip",
                showlegend=False,
            )
        )

    figure.add_trace(
        go.Scatter(
            x=[node["point"][0] for node in nodes],
            y=[node["point"][1] for node in nodes],
            mode="markers+text",
            marker=dict(color="#111827", size=9),
            text=[f"n{node['node_index']}" for node in nodes],
            textposition="top center",
            textfont=dict(color="#111827", size=11),
            hoverinfo="skip",
            showlegend=False,
        )
    )


def extract_latest_shape(final_state):
    shape_context = final_state.get("shape_context") or {}
    if isinstance(shape_context, dict) and isinstance(shape_context.get("boundary"), list):
        return shape_context

    for record in reversed(final_state.get("tool_history", [])):
        if not isinstance(record, dict):
            continue
        output = record.get("output") or {}
        data = output.get("data") if isinstance(output, dict) else None
        if isinstance(data, dict) and isinstance(data.get("boundary"), list):
            return data

    raise ValueError("No generated boundary payload was found in final_state.")


def make_decision_trace_figure(messages):
    rows = [
        message
        for message in messages
        if message.startswith(("Planner updated", "Supervisor decision", "Tool ", "Final report"))
    ]
    figure = go.Figure(
        data=[
            go.Table(
                header=dict(values=["Step", "Trace message"], fill_color="#dbeafe", align="left"),
                cells=dict(
                    values=[list(range(1, len(rows) + 1)), rows],
                    fill_color="#f8fafc",
                    align="left",
                ),
            )
        ]
    )
    figure.update_layout(title="Planner and supervisor trace")
    return figure


def make_building_graph_figure(shape_payload, *, title):
    figure = go.Figure()
    _add_centerline_graph_trace(figure, shape_payload.get("building_graph", {}))
    if not figure.data:
        raise ValueError("No drawable centerline graph was found in the supplied shape payload.")
    figure.update_layout(title=title)
    return _apply_zoom_to_fit(figure)


def make_site_plan_figure(site_boundary, *, current_shape=None, placed_buildings=None, site_objects=None, title):
    figure = go.Figure()
    _add_boundary_trace(figure, site_boundary, line_color="#1d4ed8", line_width=4)
    _add_site_object_traces(figure, site_objects)

    placed_palette = [
        ("#0f766e", "rgba(15, 118, 110, 0.18)"),
        ("#7c3aed", "rgba(124, 58, 237, 0.18)"),
        ("#c2410c", "rgba(194, 65, 12, 0.18)"),
    ]
    for index, item in enumerate(placed_buildings or []):
        if not isinstance(item, dict):
            continue
        boundary = item.get("boundary")
        if not isinstance(boundary, list):
            continue
        line_color, fill_color = placed_palette[index % len(placed_palette)]
        _add_boundary_trace(
            figure,
            boundary,
            line_color=line_color,
            line_width=3,
            fillcolor=fill_color,
        )

    if isinstance(current_shape, dict):
        _add_boundary_trace(
            figure,
            current_shape.get("boundary", []),
            line_color="#ea580c",
            line_width=3,
            fillcolor="rgba(249, 115, 22, 0.18)",
        )
        _add_centerline_graph_trace(figure, current_shape.get("building_graph", {}))

    figure.update_layout(title=title)
    return _apply_zoom_to_fit(figure)

In [ ]:
SITE_BOUNDARY = [
    [0.0, 0.0, 0.0],
    [90.0, 0.0, 0.0],
    [90.0, 60.0, 0.0],
    [0.0, 60.0, 0.0],
    [0.0, 0.0, 0.0],
]

SITE_OBJECTS = [
    {
        "name": "Main Street",
        "type": "street",
        "points": [[0.0, 8.0, 0.0], [90.0, 8.0, 0.0]],
    },
    {
        "name": "East Service Street",
        "type": "street",
        "points": [[78.0, 0.0, 0.0], [78.0, 60.0, 0.0]],
    },
    {
        "name": "Central Alignment Guide",
        "type": "alignment_guide",
        "points": [[18.0, 0.0, 0.0], [18.0, 60.0, 0.0]],
    },
]

PROMPT_SCENARIOS = [
    {
        "name": "Single building near Main Street",
        "prompt": (
            "Place one U-shaped building of about 900 square meters inside the site boundary. "
            "Keep the wing graph readable and align the building roughly with Main Street if that helps."
        ),
        "layout_payload": {
            "workflow_mode": "full",
            "site_boundary": SITE_BOUNDARY,
            "site_objects": SITE_OBJECTS,
            "target_building_count": 1,
            "building_intents": [
                "Keep the wing graph readable for later wing-level edits and street alignment requests.",
            ],
        },
    },
    {
        "name": "Two buildings with street-aware placement",
        "prompt": (
            "Place two buildings inside the site boundary. "
            "Put the first building closer to Main Street and keep its graph readable for later edits. "
            "Then place a second building after the first one, keeping site access clear and using East Service Street or the Central Alignment Guide if helpful."
        ),
        "layout_payload": {
            "workflow_mode": "full",
            "site_boundary": SITE_BOUNDARY,
            "site_objects": SITE_OBJECTS,
            "target_building_count": 2,
            "requested_positions": [[24.0, 20.0], [62.0, 34.0]],
            "building_intents": [
                "Primary building close to Main Street with a clear wing graph.",
                "Second building should fit after the first placement and stay aware of street access.",
            ],
        },
    },
]

scenario_index = 1
selected_scenario = PROMPT_SCENARIOS[scenario_index]
prompt = selected_scenario["prompt"]
layout_payload = selected_scenario["layout_payload"]

{
    "selected_scenario": selected_scenario["name"],
    "prompt": prompt,
    "target_building_count": layout_payload["target_building_count"],
    "site_object_names": [item["name"] for item in SITE_OBJECTS],
    "settings_error": settings_error,
    "llm_provider": settings.llm_provider if settings is not None else None,
    "llm_model": settings.llm_model if settings is not None else None,
}

In [ ]:
if settings is None:
    raise RuntimeError(
        "Missing LLM runtime settings. Define LLM_PROVIDER and provider credentials in the repo .env or team_04/.env before running the end-to-end agent cell."
    )

notebook_client = build_notebook_demo_tool_client(
    SITE_BOUNDARY,
    site_summary="Notebook-local site context for LangGraph node testing with site objects and optional second-building placement.",
    site_objects=SITE_OBJECTS,
    setback_m=5.0,
    spatial_intention_score=0.91,
    performance_score=0.88,
    shape_integrity_score=0.94,
 )
tool_client = CompositeToolClient([build_default_local_tool_client(), notebook_client])
catalog = ToolCatalog.from_discovered_tools(tool_client.list_tools())

llm = ChatOpenAI(
    api_key=settings.api_key,
    base_url=settings.base_url,
    model=settings.llm_model,
    timeout=settings.request_timeout_seconds,
    temperature=0,
 )
decision_engine = OpenAIDecisionEngine(
    llm=llm,
    decision_provider=settings.decision_llm_provider or settings.llm_provider,
    decision_model=settings.decision_llm_model or settings.llm_model,
    report_provider=settings.report_llm_provider or settings.llm_provider,
    report_model=settings.report_llm_model or settings.llm_model,
 )

final_state = run_agent(
    user_prompt=prompt,
    decision_engine=decision_engine,
    tool_client=tool_client,
    catalog=catalog,
    initial_layout=layout_payload,
    max_optimization_cycles=3,
    planner=RuleBasedPlanner(),
)

In [ ]:
decision_trace = [
    message
    for message in final_state.get("messages", [])
    if message.startswith(("Planner updated", "Supervisor decision", "Tool ", "Final report"))
]
placed_buildings = final_state.get("placed_buildings", [])

summary = {
    "selected_scenario": selected_scenario["name"],
    "prompt": prompt,
    "final_response": final_state.get("final_response"),
    "target_building_count": layout_payload.get("target_building_count"),
    "placed_building_count": len(placed_buildings),
    "placed_building_geometry_ids": [
        item.get("geometry_id")
        for item in placed_buildings
        if isinstance(item, dict)
    ],
    "site_object_names": [item["name"] for item in SITE_OBJECTS],
    "workflow_mode": final_state.get("workflow_mode"),
    "violations": final_state.get("violations"),
    "placement_fit_summary": final_state.get("placement_fit_summary"),
    "tool_sequence": [
        record.get("tool")
        for record in final_state.get("tool_history", [])
        if isinstance(record, dict)
    ],
    "decision_trace": decision_trace,
}

E2E_OUTPUT_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary

In [ ]:
make_decision_trace_figure(final_state.get("messages", []))

In [ ]:
latest_shape = extract_latest_shape(final_state)

make_building_graph_figure(
    latest_shape,
    title="Latest candidate centerline graph after the LangGraph node run",
)

In [ ]:
latest_shape = extract_latest_shape(final_state)

make_site_plan_figure(
    SITE_BOUNDARY,
    current_shape=latest_shape,
    placed_buildings=final_state.get("placed_buildings", []),
    site_objects=SITE_OBJECTS,
    title="Prompt-driven LangGraph node run with site objects and placed buildings",
)

In [ ]:
{
    "plan": final_state.get("plan"),
    "placed_buildings": final_state.get("placed_buildings"),
    "site_context_keys": sorted((final_state.get("site_context") or {}).keys()),
    "saved_to": str(E2E_OUTPUT_PATH),
}